# USB Camera → Shared Memory → DPU Pipeline

**Phase 1/2 smoke test** — ResNet to confirm end-to-end DPU pipeline is alive.

Flow:
```
USB Camera (PS) → raw frame → pynq.allocate buffer
  → PL fetches from shared DDR → resize/normalize
  → DPU (ResNet) → PS reads output → top-5 predictions
```

Swap `XMODEL_PATH` and call `parse_keypoints()` instead of `top5_resnet()` for MoveNet (Phase 3).

## Cell 1 — Imports & Bitstream

In [ ]:
import os
import cv2
import numpy as np
import IPython.display as display
from PIL import Image
import io

from inference_runner import load_runner, inspect_tensors, run_inference, top5_resnet

# Load bitstream + overlay (registers DPU with Linux)
try:
    from pynq import Overlay, allocate
    if os.path.exists('new.bit'):
        print('Loading overlay new.bit ...')
        ol = Overlay('new.bit')
        print('Overlay loaded.')
    else:
        print('WARNING: new.bit not found — skipping overlay load')
        allocate = None
except Exception as e:
    print(f'pynq not available or overlay failed: {e}')
    from pynq import allocate  # will fail loudly if truly missing
    allocate = None

## Cell 2 — Config

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# ResNet for Phase 1/2 smoke test. Swap for movenet.xmodel in Phase 3.
XMODEL_PATH = '/usr/share/vitis_ai_library/models/resnet50/resnet50.xmodel'
# Fallback paths to try if above doesn't exist:
# '/usr/share/vitis_ai_library/models/resnet50_pt/resnet50_pt.xmodel'
# 'models/resnet50.xmodel'

# ── Camera ────────────────────────────────────────────────────────────────────
CAMERA_INDEX = 0          # try 1 or 2 if 0 doesn't open
CAMERA_WIDTH  = 640
CAMERA_HEIGHT = 480

print(f'xmodel: {XMODEL_PATH}')
print(f'xmodel exists: {os.path.exists(XMODEL_PATH)}')

## Cell 3 — Phase 1: Open USB Camera & Capture Frame

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX)
if not cap.isOpened():
    raise RuntimeError(f'Cannot open USB camera at index {CAMERA_INDEX}. '
                       f'Run: !ls /dev/video* to check available devices.')

cap.set(cv2.CAP_PROP_FRAME_WIDTH,  CAMERA_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, CAMERA_HEIGHT)

ret, frame_bgr = cap.read()
if not ret:
    cap.release()
    raise RuntimeError('Failed to read frame from camera')

print(f'Captured frame: shape={frame_bgr.shape}  dtype={frame_bgr.dtype}')

# Display captured frame in notebook
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
img = Image.fromarray(frame_rgb)
buf = io.BytesIO()
img.save(buf, format='JPEG')
display.display(display.Image(data=buf.getvalue()))

cap.release()

## Cell 4 — Phase 1: Write Raw Frame to Shared Memory

In [ ]:
# pynq.allocate gives a physically contiguous DDR buffer the PL DMA engine can reach.
# If pynq.allocate is unavailable (off-board dev), falls back to plain numpy.

if allocate is not None:
    shared_buf = allocate(shape=frame_bgr.shape, dtype=np.uint8)
    shared_buf[...] = frame_bgr
    shared_buf.flush()   # writeback D-cache so PL DMA sees the data
    print(f'Shared memory allocated: shape={shared_buf.shape}  '
          f'physical_addr=0x{shared_buf.physical_address:08X}')
else:
    # Off-board fallback — VART will copy from normal numpy
    shared_buf = frame_bgr.copy()
    print(f'Using plain numpy buffer (no pynq): shape={shared_buf.shape}')

print('Phase 1 complete — frame in shared memory.')

## Cell 5 — Phase 2: Load VART Runner & Inspect Tensors

In [ ]:
runner = load_runner(XMODEL_PATH)
inspect_tensors(runner)

# DPU input tensor dims tell us the expected shape after PL preprocessing.
# For ResNet this is typically [1, 224, 224, 3].
# VART input buffer must match those dims exactly.
in_dims = tuple(runner.get_input_tensors()[0].dims)
print(f'\nDPU expects input shape: {in_dims}')

## Cell 6 — Phase 2: Prepare Input Buffer for DPU

PL preprocessing IP resizes and normalizes the raw frame before the DPU sees it.
On the PS side we allocate a buffer matching the DPU input dims and hand it to VART.

> **TODO (confirm with HW lead):** Does the PL preprocessing IP read directly from
> the raw-frame shared buffer (Cell 4) and write into the DPU input buffer automatically,
> or does VART/DMA orchestrate the transfer? If the IP is fully autonomous, skip this
> cell and pass `shared_buf` directly to `run_inference`.

In [ ]:
# Allocate a buffer matching DPU input dims — PL fills this after preprocessing.
# For the smoke test we just fill with dummy data to confirm the DPU fires.
# Replace with actual PL-preprocessed data once preprocessing IP is wired up.

dpu_input = np.random.randint(0, 255, size=in_dims, dtype=np.uint8)

# ── WHEN PL preprocessing is wired ───────────────────────────────────────────
# Uncomment whichever path applies after confirming with HW lead:
#
# Option A: PL IP reads raw shared_buf and writes result into dpu_input automatically.
#   trigger_pl_preprocessing(shared_buf, dpu_input)   # HW-lead provides this
#
# Option B: PS does a simple resize as a temporary stand-in (remove once PL IP works)
#   resized = cv2.resize(frame_bgr, (in_dims[2], in_dims[1]))  # (W, H)
#   dpu_input[0] = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
# ─────────────────────────────────────────────────────────────────────────────

print(f'DPU input buffer ready: shape={dpu_input.shape}  dtype={dpu_input.dtype}')

## Cell 7 — Phase 2: Run DPU Inference

In [ ]:
output_bufs = run_inference(runner, dpu_input)

print('Inference complete.')
for i, buf in enumerate(output_bufs):
    print(f'  output[{i}] shape={buf.shape}  dtype={buf.dtype}  '
          f'min={buf.min()}  max={buf.max()}  mean={buf.mean():.4f}')

## Cell 8 — Phase 2: Show Top-5 Predictions (ResNet smoke test)

In [ ]:
# Non-zero, non-uniform scores = DPU is alive and computing.
top5_resnet(output_bufs)

print('\n=== Phase 1 + 2 complete ===')
print('Camera → shared memory → DPU pipeline confirmed.')

## Cell 9 — Cleanup

In [ ]:
if hasattr(shared_buf, 'freebuffer'):
    shared_buf.freebuffer()
    print('Shared buffer freed.')

del runner

---
## Phase 3 Preview — MoveNet (swap in when Person A delivers xmodel)

```python
from inference_runner import load_runner, run_inference, parse_keypoints

XMODEL_PATH  = 'models/movenet.xmodel'
OUTPUT_SCALE = 1.0  # 1.0 if float32; set from fixpos if int8

runner      = load_runner(XMODEL_PATH)
output_bufs = run_inference(runner, dpu_input)   # dpu_input now [1,192,192,3]
keypoints   = parse_keypoints(output_bufs, output_scale=OUTPUT_SCALE)

# keypoints shape [17, 3] — [y, x, confidence] normalized [0, 1]
print(keypoints)
```